# 安全哨兵（Security Sentinel）

### 一次完成代码审计与加固

本练习对应第 4 周「用 LLM 做专项任务」：把安全审计拆成可选任务，用 **Gradio** 选任务、贴代码，再调用本地 **Ollama**（OpenAI 兼容接口）生成结果。

选择一项或多项任务，检查代码片段的安全问题：

- 扫描漏洞（Scan）
- 起草威胁情报笔记（Threat Report）
- 建议安全修复（Patch）
- 编写面向利用的测试（Exploit Test）

## 怎么跑

1. 确保本机 Ollama 已启动，并已拉取 `llama3.2:latest`
2. 从上到下依次运行单元格，最后 `ui.launch` 打开界面
3. 在界面选 Missions、粘贴代码，点 Run Audit


In [ ]:
# ========== 导入与环境：搭好后面审计流水线要用的工具 ==========

# 导入标准库 os：读环境变量、与密钥相关的进程环境打交道
import os
# 导入标准库 logging：给「哨兵」打调试/警告日志，方便排查
import logging
# 从 enum 导入 StrEnum：用字符串枚举表达提供商、任务类型（取值本身就是 str）
from enum import StrEnum
# 从 getpass 导入 getpass：在终端交互式输入密钥，避免明文写进笔记本
from getpass import getpass

# 导入 gradio：快速搭 Web 界面（Dropdown / Code / Button）
import gradio as gr
# 从 openai 导入 OpenAI：用 OpenAI 兼容协议调用本地 Ollama
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv

# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)


In [ ]:
# ========== 日志：根 logger 偏安静，本模块 logger 开 DEBUG ==========

# 配置根日志级别为 WARNING：第三方库的琐碎 INFO 不会刷屏
logging.basicConfig(level=logging.WARNING)

# 为本应用单独取名为 sentinel 的 logger
logger = logging.getLogger('sentinel')
# 把本模块日志放到 DEBUG：审计流程细节可在控制台看到
logger.setLevel(logging.DEBUG)


In [ ]:
# ========== 读密钥：环境变量 → Colab userdata → 终端 getpass ==========

# 在 Google Colab 里尝试从 userdata 取密钥；本地/非 Colab 则静默失败返回空串
def get_secret_in_google_colab(env_name: str) -> str:
    try:
      # 仅 Colab 环境存在的 userdata API
      from google.colab import userdata
      # 按名字取出密钥字符串
      return userdata.get(env_name)
    except Exception:
      # 导入失败或读不到：当作没有密钥
      return ''


# 统一入口：按优先级拿密钥，拿不到就交互式询问
def get_secret(env_name: str) -> str:
    # 先看进程环境，再看 Colab；任一非空即可
    key = os.environ.get(env_name) or get_secret_in_google_colab(env_name)

    # 两边都没有：用 getpass 在终端提示用户粘贴（不回显）
    if not key:
        key = getpass(f'Enter {env_name}:').strip()

    # 有值打 info，没有打 warning（便于确认配置是否就绪）
    if key:
        logger.info(f'✅ {env_name} provided')
    else:
        logger.warning(f'❌ {env_name} not provided')
    # 返回去掉首尾空白的密钥（可能仍是空串）
    return key.strip()


In [ ]:
# ========== 模型客户端：本练习只用本地 Ollama（OpenAI 兼容 /v1）==========

# 提供商枚举：目前只登记 Ollama
class Provider(StrEnum):
    OLLAMA = 'Ollama'

# 客户端字典：键是 Provider，值是 OpenAI 兼容客户端
clients: dict[Provider, OpenAI] = {}

# 指向本机 Ollama 的 OpenAI 兼容基址（默认 11434 端口）
clients[Provider.OLLAMA] = OpenAI(base_url='http://localhost:11434/v1')

# 要调用的模型名：须与本机 ollama list 中的名字一致
model = 'llama3.2:latest'
# 取出 Ollama 客户端；后面 chat.completions 都用它
client = clients.get(Provider.OLLAMA)
# 取不到就立刻失败，避免后面静默空指针
if not client:
    raise Exception('No client found')


In [ ]:
# ========== 核心：按勾选任务拼 system prompt，再调 chat.completions ==========

# 任务类型枚举：界面 Dropdown 的选项值就是这些字符串
class Task(StrEnum):
  SCAN = 'Scan'
  REPORT = 'Threat Report'
  PATCH = 'Patch'
  TEST = 'Exploit Test'


# Gradio 回调：tasks 是勾选的任务列表，code 是待审计源码
def perform_tasks(tasks, code):
  # 记录本次要执行的任务，便于调试
  logger.info(f'Performing tasks: {tasks}')

  # steps：把用户勾选映射成给模型看的英文指令句（prompt 原文勿改）
  steps = []
  if Task.SCAN in tasks:
    steps.append('Scan the snippet for security weaknesses and name them clearly.')
  if Task.REPORT in tasks:
    steps.append('Produce a concise threat report that explains impact, likelihood, and affected components.')
  if Task.PATCH in tasks:
    steps.append('Propose hardened code that mitigates the identified risks without changing intent.')
  if Task.TEST in tasks:
    steps.append('Design exploit-style tests or probing steps that would validate the vulnerability.')

  # 拼成无序列表；若什么都没选，给一句占位说明
  task_list = '- ' + '\n- '.join(steps) if steps else 'No security directive selected.'
  # system prompt：角色 + 任务列表（动态插入 task_list）
  system_prompt = f"""
  You are a seasoned application security engineer who recognises languages instantly and
  maps weaknesses to common vulnerability classes.
  Only rewrite code when instructed to patch it.

  Your tasks:
  {task_list}
  """
  # messages：system 定行为，user 放待审代码
  messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": f'Code: \n{code}'}
  ]
  # 调用本地模型；model / messages 逻辑保持原样
  response =  client.chat.completions.create(
      model=model,
      messages=messages
  )

  # 取出助手回复正文
  content = response.choices[0].message.content

  # 返回给 Gradio Markdown 输出框
  return content


In [ ]:
# ========== 示例库：给 Gradio Examples 预置多语言脆弱片段 ==========

# 返回 (examples 行列表, 每行标签)；供界面一键填入
def get_examples() -> tuple[list[any], list[str]]:
    # Python：字符串拼 SQL，典型注入面
    python_sql = r'''
    import sqlite3

    def get_user(conn, user_id):
        query = f"SELECT * FROM users WHERE id = {user_id}"
        cursor = conn.cursor()
        cursor.execute(query)
        return cursor.fetchone()
    '''

    # JavaScript：只 decode JWT、未验签就查库
    js_auth = r'''
    app.post('/login', async (req, res) => {
        const token = jwt.decode(req.body.token);
        if (!token) {
            return res.status(401).send('blocked');
        }
        const user = await db.find(token.user);
        res.send(user);
    });
    '''

    # Go：密码用 SHA1，哈希强度不足
    go_crypto = r'''
    package main

    import (
        "crypto/sha1"
    )

    func hashPassword(password string) string {
        h := sha1.New()
        h.Write([]byte(password))
        return string(h.Sum(nil))
    }
    '''

    # PHP：上传文件名直接拼路径，易被路径穿越/覆盖
    php_upload = r'''
    <?php
    if ($_FILES["file"]["error"] == 0) {
        move_uploaded_file($_FILES["file"]["tmp_name"], "/uploads/" . $_FILES["file"]["name"]);
        echo "done";
    }
    ?>
    '''

    # Rust：环境变量缺失会 panic；SERVICE_URL 暴露面
    rust_config = r'''
    use std::env;

    fn main() {
        let endpoint = env::var("SERVICE_URL").unwrap();
        println!("Connecting to {}", endpoint);
    }
    '''

    # 每一行：[任务列表, 代码, 语言标签] —— 与 Examples 的 inputs 对齐
    examples = [
        [[Task.SCAN], python_sql, 'python'],
        [[Task.REPORT], js_auth, 'javascript'],
        [[Task.PATCH], go_crypto, 'go'],
        [[Task.TEST], php_upload, 'php'],
        [[Task.SCAN, Task.PATCH, Task.REPORT], rust_config, 'rust']
    ]

    # 界面上显示的人类可读标签（英文原文保留）
    example_labels = [
        'Python: SQL injection review',
        'JavaScript: Token handling report',
        'Go: Strengthen hashing',
        'PHP: Exploit upload path',
        'Rust: Exposure analysis'
    ]

    return examples, example_labels


## 界面

下面用 **Gradio Blocks** 拼左右栏：左侧选任务 + 贴代码，右侧看 Findings；底部 Examples 可一键载入示例并触发语法高亮。


In [ ]:
# ========== Gradio UI：任务多选 + 代码编辑 + 示例 + 审计按钮 ==========

# 浏览器标签页标题
title = 'Security Sentinel'

# 创建 Blocks 应用；Monochrome 主题偏简约灰阶
with gr.Blocks(title=title, theme=gr.themes.Monochrome()) as ui:
    # 页头标题
    gr.Markdown(f'# {title}')
    # 副标题说明（界面文案保持英文原样）
    gr.Markdown('## Run rapid security sweeps on any snippet.')

    # 左右两列布局
    with gr.Row():
      with gr.Column():
        # 多选下拉：choices 来自 Task 枚举的 value
        tasks = gr.Dropdown(
            label="Missions",
            choices=[task.value for task in Task],
            value=Task.SCAN,
            multiselect=True,
            interactive=True,
        )
        # 代码输入框；lines=40 给足编辑高度
        code_input = gr.Code(
            label='Code Input',
            lines=40,
        )
        # 隐藏文本框：承载示例里的语言字符串，供 set_language 读
        code_language = gr.Textbox(visible=False)

      with gr.Column():
        # 右侧标题
        gr.Markdown('## Findings')
        # 审计结果用 Markdown 渲染
        code_output = gr.Markdown('Awaiting report')


    # 触发审计的主按钮
    run_btn = gr.Button('Run Audit')

    # Examples 点击时：根据语言字符串刷新 Code 组件的语法高亮
    def set_language(tasks, code, language):
      # Gradio Code 支持的高亮语言白名单
      syntax_highlights = ['python', 'c', 'cpp', 'javascript', 'typescript', 'go', 'rust', 'php']
      logger.debug(f'Tasks: {tasks}, Language: {language}')
      # 不在白名单则不高亮
      highlight = language if language in syntax_highlights else None

      # 返回：任务原样 + 带 language 的 Code 更新
      return tasks, gr.Code(value=code, language=highlight)

    # 取出示例数据与标签
    examples, example_labels = get_examples()
    # 绑定 Examples：点选即填入并跑 set_language
    examples = gr.Examples(
        examples=examples,
        example_labels=example_labels,
        examples_per_page=20,
        inputs=[tasks, code_input, code_language],
        outputs=[tasks, code_input],
        run_on_click=True,
        fn=set_language
    )

    # 主按钮：调用 perform_tasks，结果写到 code_output
    run_btn.click(perform_tasks, inputs=[tasks, code_input], outputs=[code_output])

# 启动本地 Gradio；debug=True 便于看报错栈
ui.launch(debug=True)
